# Financial Market Data Platform: 02. Market Analytics & Risk Engine
### Volatility Modeling, Drawdown Profiling, Cross-Asset Correlations & Portfolio Attribution

**Author:** Marouane Bizou  
**Role:** Senior Data Engineer / Quantitative Analyst  
**Focus:** Time-Series Analytics, Drawdowns, VaR / CVaR, Euler Risk Decomposition

---
## 1. Overview
This notebook executes in-depth financial analytics on the **Gold Feature Layer** (`data/gold/gold_market_features.parquet`):
1. **Cumulative Performance Trajectories:** Performance across market cycles (2021–2025).
2. **Volatility Dynamics:** Close-to-Close vs. Parkinson High-Low range estimators.
3. **Drawdown Anatomy:** Peak-to-trough coordinate identification, recovery duration, and maximum drawdowns.
4. **Cross-Asset Correlations:** Heatmaps and rolling correlations highlighting flight-to-safety regime shifts.
5. **Portfolio Risk Attribution:** Euler Component Contribution to Risk (CCR) for multi-asset portfolios.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import GOLD_DATA_DIR, load_assets_config
from src.analytics.returns import calculate_cagr
from src.analytics.volatility import calculate_annualized_volatility, calculate_parkinson_volatility
from src.analytics.drawdown import analyze_maximum_drawdown
from src.analytics.risk import calculate_portfolio_performance, calculate_var_historical, calculate_cvar

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Libraries and analytics modules loaded.")


## 2. Load Gold Layer Features & Pivot Price Series


In [ ]:
df_gold = pd.read_parquet(GOLD_DATA_DIR / "gold_market_features.parquet")
prices_pivot = df_gold.pivot(index="date", columns="symbol", values="adjusted_close")
returns_pivot = df_gold.pivot(index="date", columns="symbol", values="daily_return").dropna()

print(f"Analyzed {prices_pivot.shape[1]} instruments across {prices_pivot.shape[0]} trading sessions.")


## 3. Cumulative Growth of $1.00 Across Asset Classes
Compare the compounded growth trajectories across Equities (NVDA, AAPL), Broad Market ETFs (SPY, QQQ), Commodities (GLD), and Fixed Income (TLT).


In [ ]:
benchmarks = ["SPY", "QQQ", "GLD", "TLT", "NVDA", "AAPL"]
cum_returns = (1.0 + returns_pivot[benchmarks]).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
for sym in benchmarks:
    ax.plot(cum_returns.index, cum_returns[sym], label=sym, linewidth=2)

ax.set_title("Cumulative Growth of $1.00 (2021 - 2025)", fontsize=14, fontweight="bold")
ax.set_ylabel("Growth Factor ($1.00 base)")
ax.set_yscale("log")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()


## 4. Volatility Modeling: Close-to-Close vs. Parkinson High-Low Estimator
The **Parkinson Estimator** leverages intraday price extremities $(High / Low)$ rather than discrete closing snapshots, providing up to 5x higher statistical efficiency.


In [ ]:
spy_data = df_gold[df_gold["symbol"] == "SPY"].copy().set_index("date")
spy_data["parkinson_vol_20d"] = calculate_parkinson_volatility(spy_data["high"], spy_data["low"], window=20)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(spy_data.index, spy_data["volatility_20d"] * 100, label="Standard Rolling Volatility (20D)", color="#1f77b4")
ax.plot(spy_data.index, spy_data["parkinson_vol_20d"] * 100, label="Parkinson Range Volatility (20D)", color="#ff7f0e", linestyle="--")

ax.set_title("SPY: Standard Close-to-Close vs. Parkinson Intraday Volatility", fontsize=14, fontweight="bold")
ax.set_ylabel("Annualized Volatility (%)")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Peak-to-Trough Drawdown Analysis
We compute the continuous drawdown series from historical peaks and determine the deepest drawdown coordinates for each asset.


In [ ]:
dd_summary = []
for sym in prices_pivot.columns:
    dd_info = analyze_maximum_drawdown(prices_pivot[sym])
    dd_summary.append({
        "symbol": sym,
        "max_drawdown_pct": dd_info["max_drawdown"] * 100,
        "peak_date": dd_info["peak_date"],
        "trough_date": dd_info["trough_date"],
        "duration_trading_days": dd_info["drawdown_duration"]
    })

df_dd = pd.DataFrame(dd_summary).sort_values(by="max_drawdown_pct")
df_dd.head(10)


## 6. Cross-Asset Correlation Matrix
Evaluating co-movement between Equities, Tech, Gold, and Long-Term Treasuries.


In [ ]:
corr_matrix = returns_pivot.corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, cbar_kws={'label': 'Pearson Correlation'})
ax.set_title("Cross-Asset Daily Return Correlation Matrix (2021 - 2025)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Multi-Asset Portfolio Risk Decomposition (Euler CCR)
We evaluate the balanced model portfolio (`35% SPY, 25% QQQ, 20% GLD, 20% TLT`) and attribute total volatility using Euler's Theorem:
$$\sigma_p = \sum_{i=1}^N w_i rac{\partial \sigma_p}{\partial w_i}$$


In [ ]:
config = load_assets_config()
weights = config["portfolio"]["weights"]

port_results = calculate_portfolio_performance(returns_pivot, weights)
print("=== BALANCED MODEL PORTFOLIO PERFORMANCE ===")
print(f"Annualized CAGR:        {port_results['portfolio_cagr'] * 100:.2f}%")
print(f"Annualized Volatility:  {port_results['annualized_volatility'] * 100:.2f}%")
print(f"Sharpe Ratio (Rf=3.5%): {port_results['sharpe_ratio']:.2f}")
print(f"Sortino Ratio:          {port_results['sortino_ratio']:.2f}")
print(f"Maximum Drawdown:       {port_results['max_drawdown'] * 100:.2f}%")
print(f"1-Day 95% Historical VaR: {port_results['var_95_daily'] * 100:.2f}%")
print(f"1-Day 95% CVaR:         {port_results['cvar_95_daily'] * 100:.2f}%")

df_risk = pd.DataFrame(port_results["risk_decomposition"]).T
df_risk
